In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [2]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [3]:
train_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
train_data.info()
train_data.describe()
train_data.isnull().sum() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
numerical_columns = train_data.select_dtypes(include=['float64', 'int64']).columns
categorical_columns = train_data.select_dtypes(exclude=['float64', 'int64']).drop(columns=["Transported"]).columns

In [6]:
train_data[numerical_columns] = train_data[numerical_columns].fillna(train_data[numerical_columns].mean())
test_data[numerical_columns] = test_data[numerical_columns].fillna(test_data[numerical_columns].mean())

In [7]:
# Замена пропущенных значений в категориальных столбцах наиболее частым значением
train_data[categorical_columns] = train_data[categorical_columns].fillna(train_data[categorical_columns].mode().iloc[0]).astype(str).infer_objects(copy=False)
test_data[categorical_columns] = test_data[categorical_columns].fillna(test_data[categorical_columns].mode().iloc[0]).astype(str).infer_objects(copy=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_20440\1502401430.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_data[categorical_columns] = train_data[categorical_columns].fillna(train_data[categorical_columns].mode().iloc[0]).astype(str).infer_objects(copy=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_20440\1502401430.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_data[categorical_columns] = test_data[categorical_columns].fillna(test_data[categorical_columns].mode().iloc[0]).astype(str).infer_objects(copy=False)


In [8]:
# Объединяем тренировочные и тестовые данные для правильного кодирования категорий
combined_data = pd.concat([train_data[categorical_columns], test_data[categorical_columns]])

In [9]:
# Преобразование категориальных признаков с помощью LabelEncoder
# Теперь обрабатываем каждую колонку отдельно
encoded_categorical_columns = []
for cat_column in categorical_columns:
    label_encoder = LabelEncoder()
    combined_column = pd.concat([train_data[cat_column], test_data[cat_column]])  # Объединяем тренировочные и тестовые данные
    encoded_column = label_encoder.fit_transform(combined_column)
    encoded_categorical_columns.append(pd.Series(encoded_column))


In [10]:
# Разделяем обратно на тренировочные и тестовые данные
encoded_train_data = [encoded_column[:len(train_data)] for encoded_column in encoded_categorical_columns]
encoded_test_data = [encoded_column[len(train_data):] for encoded_column in encoded_categorical_columns]

In [11]:
# Объединение числовых и категориальных признаков
encoded_train_data = pd.concat([pd.DataFrame(encoded_train_data), train_data[numerical_columns]], axis=1)
encoded_test_data = pd.concat([pd.DataFrame(encoded_test_data), test_data[numerical_columns]], axis=1)

In [12]:
# Определение целевой переменной и признаков
X_train = encoded_train_data
y_train = train_data["Transported"].values.ravel()  # Извлекаем целевой вектор как одномерный массив
X_test = encoded_test_data

In [13]:
# Преобразование типов имен столбцов в строки
X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

In [14]:
# Объединяем тренировочные и тестовые данные перед стандартизацией
combined_data = pd.concat([X_train, X_test], ignore_index=True)

In [15]:
# Стандартизация данных
scaler = StandardScaler()
scaled_data = scaler.fit_transform(combined_data)

In [16]:
# Разделение обратно на тренировочные и тестовые данные
X_train_scaled = scaled_data[:len(X_train)]
X_test_scaled = scaled_data[len(X_train):]

In [17]:
# Построение модели
model = DecisionTreeClassifier()
model.fit(X_train_scaled, y_train)

DecisionTreeClassifier()

In [18]:
# Предсказание на тестовых данных
predictions = model.predict(X_test_scaled)

In [19]:
# Сохранение результатов
results = pd.DataFrame({"PassengerId": test_data["PassengerId"], "Predicted_Transported": predictions})
results.to_csv("predicted_results.csv", index=False)